# 🚀 UniXcoder Final Training Notebook

This notebook implements a streamlined, reproducible training pipeline for **UniXcoder (text-only baseline)**.


In [1]:
!pip install torch transformers scikit-learn tqdm

In [2]:
import os
import json
import torch
import logging
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler, random_split
from torch.optim import AdamW
from transformers import (get_linear_schedule_with_warmup, 
                          RobertaConfig, RobertaModel, AutoTokenizer)
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score, 
    roc_auc_score, average_precision_score, 
    classification_report, confusion_matrix
)

# --- CONFIGURATION ---
class Args:
    output_dir = "saved_models_unixcoder"
    model_name_or_path = "microsoft/unixcoder-base"
    
    # Data Path
    train_file = "/kaggle/input/datasets/hasanmahmudabdullah/dfgdataset2/dataset_graphcodebert.jsonl"
    
    # Hyperparameters
    code_length = 384 # Standardized with GraphCodeBERT for fair comparison
    train_batch_size = 16
    eval_batch_size = 32
    learning_rate = 2e-5
    max_grad_norm = 1.0
    num_train_epochs = 3
    seed = 42
    
    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

args = Args()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(args.seed)


In [3]:
class SimpleModel(nn.Module):   
    def __init__(self, encoder, config):
        super(SimpleModel, self).__init__()
        self.encoder = encoder
        self.config = config
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None): 
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0] # [Batch, Seq, Hidden]
        
        # Use CLS token for classification
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob = F.softmax(logits, dim=-1)

        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            return loss, prob
        return prob

In [4]:
class SimpleDataset(Dataset):
    def __init__(self, tokenizer, args, file_path):
        self.args = args
        self.tokenizer = tokenizer
        
        with open(file_path, 'r') as f:
            self.lines = f.readlines()
            
    def __len__(self):
        return len(self.lines)

    def __getitem__(self, item):
        line = self.lines[item]
        entry = json.loads(line)
        
        code = entry.get('code', '')
        label = int(entry.get('label', 0)) if entry.get('label') is not None else 0

        tokens_obj = self.tokenizer(
            code, 
            max_length=self.args.code_length, 
            truncation=True, 
            padding='max_length'
        )
        
        return {
            'input_ids': torch.tensor(tokens_obj['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(tokens_obj['attention_mask'], dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [5]:
tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path)
full_dataset = SimpleDataset(tokenizer, args, args.train_file)

total = len(full_dataset)
train_size = int(0.90 * total)
test_size  = total - train_size
generator  = torch.Generator().manual_seed(42)
train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size], generator=generator
)

print(f"Dataset Split Results:")
print(f"- Total: {total}")
print(f"- Train: {train_size} (90%)")
print(f"- Test:  {test_size} (10%)")


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/vocab.json "HTTP/1.1 200 OK"


...

merges.txt: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Dataset Split Results:
- Total: 199960
- Train: 179964 (90%)
- Test:  19996 (10%)


In [6]:
def train(model, train_dataset, args):
    train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=args.train_batch_size, num_workers=2, pin_memory=True)
    
    optimizer = AdamW(model.parameters(), lr=args.learning_rate, eps=1e-8)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_dataloader) * args.num_train_epochs)
    scaler = GradScaler()
    
    if not os.path.exists(args.output_dir): os.makedirs(args.output_dir)

    for epoch in range(args.num_train_epochs):
        model.train()
        tr_loss = 0
        bar = tqdm(train_dataloader, desc=f"Epoch {epoch}")
        for step, batch in enumerate(bar):
            optimizer.zero_grad()
            with autocast():
                loss, _ = model(
                    input_ids=batch['input_ids'].to(args.device), 
                    p_ids=batch['p_ids'].to(args.device) if 'p_ids' in batch else None, 
                    attn_mask=batch.get('attn_mask', batch.get('attention_mask')).to(args.device), 
                    labels=batch['label'].to(args.device)
                ) if 'p_ids' in batch else model(
                    input_ids=batch['input_ids'].to(args.device), 
                    attention_mask=batch.get('attn_mask', batch.get('attention_mask')).to(args.device), 
                    labels=batch['label'].to(args.device)
                )
            
            scaler.scale(loss).backward()
            
            # --- Gradient Clipping ---
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss += loss.item()
            bar.set_postfix(loss=tr_loss/(step+1))

    # After the epoch loop
    torch.save(model.state_dict(), os.path.join(args.output_dir, "model.bin"))
    print(f"Model saved to {os.path.join(args.output_dir, 'model.bin')}")


def evaluate(model, dataset, args, tag="Test"):
    dataloader = DataLoader(
        dataset,
        sampler=SequentialSampler(dataset),
        batch_size=args.eval_batch_size,
        num_workers=2,
        pin_memory=True
    )
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {tag}"):
            probs = model(
                input_ids=batch['input_ids'].to(args.device),
                attention_mask=batch['attention_mask'].to(args.device)
            )
            all_probs.append(probs.cpu().numpy())
            all_labels.extend(batch['label'].cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.argmax(all_probs, axis=-1)

    acc     = accuracy_score(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs[:, 1])
    pr_auc  = average_precision_score(all_labels, all_probs[:, 1])
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    print("\n" + "=" * 40)
    print(f"FINAL TEST RESULTS ({tag})")
    print("=" * 40)
    print(f"Accuracy : {acc:.4%}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")
    print(f"FN Count : {fn}  (missed vulnerabilities)")
    print(f"FP Count : {fp}  (false alarms)")
    print("-" * 40)
    print(classification_report(all_labels, all_preds, target_names=['Safe', 'Vuln'], digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return acc, all_probs, np.array(all_labels)


In [7]:
# Initialization
config = RobertaConfig.from_pretrained(args.model_name_or_path)
config.num_labels = 2
encoder = RobertaModel.from_pretrained(args.model_name_or_path, config=config)
model = SimpleModel(encoder, config)
model.to(args.device)

# Train on full 90% split
train(model, train_dataset, args)

# Load final checkpoint and evaluate once on the held-out 10% test set
model.load_state_dict(torch.load(os.path.join(args.output_dir, "model.bin")))
acc, probs, labels = evaluate(model, test_dataset, args, tag="UniXcoder")

# Save raw outputs for downstream analysis
np.save("/kaggle/working/test_probs.npy", probs)
np.save("/kaggle/working/test_labels.npy", labels)

# Write summary metrics to text file
preds   = np.argmax(probs, axis=-1)
roc_auc = roc_auc_score(labels, probs[:, 1])
pr_auc  = average_precision_score(labels, probs[:, 1])
tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

out_path = "/kaggle/working/unixcoder_results.txt"
with open(out_path, "w") as f:
    f.write(f"Split        : 90/10 train/test (no validation)\n")
    f.write(f"Epochs       : {args.num_train_epochs}\n")
    f.write(f"Accuracy     : {acc:.4%}\n")
    f.write(f"ROC-AUC      : {roc_auc:.4f}\n")
    f.write(f"PR-AUC       : {pr_auc:.4f}\n")
    f.write(f"FN           : {fn}\n")
    f.write(f"FP           : {fp}\n")
print(f"Saved results to {out_path}")


INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/unixcoder-base/5604afdc964f6c53782a6813140ade5216b99006/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/discussions?p=0 "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: microsoft/unixcoder-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/commits/refs%2Fpr%2F8 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/unixcoder-base/resolve/refs%2Fpr%2F8/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/unixcoder-base/xet-read-token/558aa506226b13a7e83f66cb38c53e61b9603eac "HTTP/1

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

/tmp/ipykernel_25/123939245.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()

Epoch 0:   0%|          | 0/11248 [00:00<?, ?it/s]/tmp/ipykernel_25/123939245.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():

Epoch 2: 100%|██████████| 11248/11248 [1:02:59<00:00,  2.98it/s, loss=0.19]


Model saved to saved_models_unixcoder/model.bin


Evaluating UniXcoder: 100%|██████████| 625/625 [07:53<00:00,  1.32it/s]


FINAL TEST RESULTS (UniXcoder)
Accuracy : 89.2829%
ROC-AUC  : 0.9652
PR-AUC   : 0.9657
FN Count : 1051  (missed vulnerabilities)
FP Count : 1092  (false alarms)
----------------------------------------
              precision    recall  f1-score   support

        Safe     0.8955    0.8918    0.8936     10095
        Vuln     0.8902    0.8938    0.8920      9901

    accuracy                         0.8928     19996
   macro avg     0.8928    0.8928    0.8928     19996
weighted avg     0.8928    0.8928    0.8928     19996

Confusion Matrix:
[[9003 1092]
 [1051 8850]]
Saved results to /kaggle/working/unixcoder_results.txt
